# Chapter 39 — Efficient Fine-Tuning: LoRA and Adapters

*From Absolute Zero* — companion notebook.

Every block below is the code printed in the chapter, in the same order. Run the cells top to bottom; the output should match the book exactly. If it does not, check `requirements.txt` first, then the errata page.

In [ ]:
!pip -q install -r https://raw.githubusercontent.com/USER/from-absolute-zero/main/requirements.txt  # Colab only; skip locally

## Shared setup

Imports and the objects the blocks below reuse. The chapter prints these once and then continues the same session.

In [ ]:
import numpy as np, warnings; warnings.filterwarnings("ignore")

def softmax(z):
    z = z - z.max(axis=-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)

## The chapter code

### Block 1  (`c1.py`)

In [ ]:
# A controlled setting: a large pretrained weight matrix, of the kind a
# transformer's attention or feedforward projection actually uses, and
# a target task whose correct adaptation is, by construction, a
# low-rank change. This lets the low-rank hypothesis behind LoRA be
# checked directly rather than only argued for.
r = np.random.default_rng(39)
D = 256                                   # a modest transformer-layer width
W_pretrained = r.normal(0, np.sqrt(1/D), (D, D))

true_rank = 4
A_true = r.normal(0, 0.5, (D, true_rank))
B_true = r.normal(0, 0.5, (true_rank, D))
W_target_true = W_pretrained + A_true @ B_true      # the update the target task actually needs

full_params = W_pretrained.size
lora_params = D * true_rank + true_rank * D

print(f"pretrained weight matrix:      {W_pretrained.shape}   {full_params:,} parameters")
print(f"true adaptation needed:       rank {true_rank}")
print(f"full fine-tuning would train: {full_params:,} parameters")
print(f"LoRA at rank {true_rank} would train:  {lora_params:,} parameters "
      f"({100*lora_params/full_params:.2f}% of full)")
print(f"that is a {full_params/lora_params:.0f}x reduction in trainable parameters")

### Block 2  (`c2.py`)

In [ ]:
# A regression framing: the loss is direct mean-squared error between
# the current matrix's hidden representation and the target
# transformation's hidden representation, on unlabelled input vectors.
# This is closer to how LoRA is actually posed, and it removes the
# classification-boundary noise that would otherwise obscure what the
# rank itself is doing.
def make_inputs(n, seed):
    return np.random.default_rng(seed).normal(size=(n, D))

Xtr = make_inputs(2000, seed=101)
Xte = make_inputs(500, seed=102)
Htr_target = np.tanh(Xtr @ W_target_true)          # what the target task actually wants
Hte_target = np.tanh(Xte @ W_target_true)

def mse_loss(W, X, H_target):
    H = np.tanh(X @ W)
    return np.mean((H - H_target) ** 2)

def train_full(Xtr, Htr_target, Xte, Hte_target, W_init, epochs=300, eta=1.0):
    W = W_init.copy()
    for _ in range(epochs):
        H = np.tanh(Xtr @ W)
        dH = 2 * (H - Htr_target) * (1 - H**2) / len(Xtr)
        dW = Xtr.T @ dH
        W -= eta * dW
    return W, mse_loss(W, Xte, Hte_target)

loss_frozen = mse_loss(W_pretrained, Xte, Hte_target)
W_full, loss_full = train_full(Xtr, Htr_target, Xte, Hte_target, W_pretrained)

print(f"held-out MSE, unadapted pretrained matrix: {loss_frozen:.4f}")
print(f"held-out MSE, after FULL fine-tuning ({full_params:,} params): {loss_full:.6f}")
print(f"\n(a perfect match to the target transformation gives MSE 0)")

### Block 3  (`c3.py`)

In [ ]:
# LoRA: freeze the pretrained matrix entirely and train only a low-rank
# update A @ B added on top of it. The rank r controls exactly how many
# parameters this costs: D*r for A, plus r*D for B.
def train_lora(Xtr, Htr_target, Xte, Hte_target, W_frozen, rank, seed, epochs=300, eta=1.0):
    rr = np.random.default_rng(seed)
    A = rr.normal(0, 0.01, (D, rank))          # small init, as the LoRA paper prescribes
    B = np.zeros((rank, D))                     # B starts at zero: the update starts at zero
    for _ in range(epochs):
        W = W_frozen + A @ B
        H = np.tanh(Xtr @ W)
        dH = 2 * (H - Htr_target) * (1 - H**2) / len(Xtr)
        dW = Xtr.T @ dH                          # gradient w.r.t. the full update
        dA = dW @ B.T
        dB = A.T @ dW
        A -= eta * dA; B -= eta * dB
    W_final = W_frozen + A @ B
    return A, B, mse_loss(W_final, Xte, Hte_target)

print(f"{'rank':>6}{'trainable params':>18}{'% of full':>12}{'held-out MSE':>14}")
for rank in (1, 2, 4, 8, 16, 32):
    n_params = D * rank * 2
    _, _, loss = train_lora(Xtr, Htr_target, Xte, Hte_target, W_pretrained,
                            rank=rank, seed=39)
    print(f"{rank:>6}{n_params:>18,}{100*n_params/full_params:>12.2f}{loss:>14.6f}")

print(f"\nfor reference: unadapted {loss_frozen:.4f}, full fine-tune {loss_full:.6f} "
      f"({full_params:,} params)")

### Block 4  (`c4.py`)

In [ ]:
# Does the same pattern hold on a real task, not just a constructed one?
# Reuse Chapter 35's exact source/target digit split: a CNN pretrained
# on digits zero through four, adapted to digits five through nine.
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from numpy.lib.stride_tricks import sliding_window_view

digits = load_digits()
X_img = digits.images / 16.0
y_all = digits.target
src_mask, tgt_mask = y_all < 5, y_all >= 5
Xsrc, ysrc = X_img[src_mask], y_all[src_mask]
Xtgt, ytgt = X_img[tgt_mask], y_all[tgt_mask] - 5
Xsrc_tr, Xsrc_te, ysrc_tr, ysrc_te = train_test_split(Xsrc, ysrc, test_size=0.2,
                                                      stratify=ysrc, random_state=0)
Xtgt_tr, Xtgt_te, ytgt_tr, ytgt_te = train_test_split(Xtgt, ytgt, test_size=0.3,
                                                      stratify=ytgt, random_state=0)

def conv_forward(imgs, filters):
    windows = sliding_window_view(imgs, (3, 3), axis=(1, 2))
    return np.einsum('nijhw,fhw->nfij', windows, filters), windows

def pool_forward(feats, size=2):
    n, nf, h, w = feats.shape
    rr = feats.reshape(n, nf, h // size, size, w // size, size)
    return rr.max(axis=(3, 5))

def extract_features(imgs, filters):
    c, _ = conv_forward(imgs, filters)
    return pool_forward(np.maximum(0, c)).reshape(len(imgs), -1)

# pretrain filters on the source task (identical recipe to Chapter 35)
def train_cnn_source(Xtr, ytr, seed, n_filters=4, epochs=150):
    rr = np.random.default_rng(seed)
    filters = rr.normal(0, np.sqrt(2/9), (n_filters, 3, 3))
    D_flat = n_filters * 9
    Wf = rr.normal(0, np.sqrt(2/D_flat), (D_flat, 5)); bf = np.zeros(5)
    Y = np.eye(5)[ytr]
    for _ in range(epochs):
        c, windows = conv_forward(Xtr, filters)
        relu = np.maximum(0, c)
        pool = pool_forward(relu).reshape(len(Xtr), -1)
        p = softmax(pool @ Wf + bf)
        dscore = (p - Y) / len(Xtr)
        Wf -= 0.3 * (pool.T @ dscore); bf -= 0.3 * dscore.sum(0)
    return filters

src_filters = train_cnn_source(Xsrc_tr, ysrc_tr, seed=39)
D_feat = src_filters.size // 9 * 9         # 36, the pooled-feature dimension
feat_tr = extract_features(Xtgt_tr, src_filters)
feat_te = extract_features(Xtgt_te, src_filters)

def train_head_full(feat_tr, ytr, feat_te, yte, seed, epochs=200, eta=0.5):
    rr = np.random.default_rng(seed)
    D_ = feat_tr.shape[1]
    W = rr.normal(0, np.sqrt(1/D_), (D_, 5)); b = np.zeros(5)
    Y = np.eye(5)[ytr]
    for _ in range(epochs):
        p = softmax(feat_tr @ W + b)
        dscore = (p - Y) / len(feat_tr)
        W -= eta * (feat_tr.T @ dscore); b -= eta * dscore.sum(0)
    return (softmax(feat_te @ W + b).argmax(1) == yte).mean(), W.size

def train_head_lora(feat_tr, ytr, feat_te, yte, seed, rank, epochs=400, eta=0.1):
    rr = np.random.default_rng(seed)
    D_ = feat_tr.shape[1]
    W0 = rr.normal(0, np.sqrt(1/D_), (D_, 5))       # frozen "pretrained" head init
    A = rr.normal(0, 0.01, (D_, rank)); B = np.zeros((rank, 5))
    b = np.zeros(5)
    Y = np.eye(5)[ytr]
    for _ in range(epochs):
        W = W0 + A @ B
        p = softmax(feat_tr @ W + b)
        dscore = (p - Y) / len(feat_tr)
        dW = feat_tr.T @ dscore
        A -= eta * (dW @ B.T); B -= eta * (A.T @ dW); b -= eta * dscore.sum(0)
    W = W0 + A @ B
    return (softmax(feat_te @ W + b).argmax(1) == yte).mean(), A.size + B.size

acc_full, params_full = train_head_full(feat_tr, ytgt_tr, feat_te, ytgt_te, seed=39)
print(f"real digit task, full fine-tune of the head: {acc_full:.4f}  ({params_full} params)")
for rank in (1, 2, 4, 8):
    acc_lora, params_lora = train_head_lora(feat_tr, ytgt_tr, feat_te, ytgt_te, seed=39, rank=rank)
    print(f"real digit task, LoRA rank {rank}:            {acc_lora:.4f}  "
          f"({params_lora} params, {100*params_lora/params_full:.0f}% of full)")

### Block 5  (`c5.py`)

In [ ]:
# An adapter takes a different architectural approach: rather than
# adding a low-rank update to an existing weight matrix, insert a small
# bottleneck layer after it, and train only that. Applied to the same
# large pretrained matrix from Steps 1-3, at a comparable parameter
# budget to the rank-4 LoRA above.
def train_adapter(Xtr, Htr_target, Xte, Hte_target, W_frozen, bottleneck, seed,
                  epochs=500, eta=0.01):
    rr = np.random.default_rng(seed)
    W_down = rr.normal(0, np.sqrt(1/D), (D, bottleneck))
    W_up = np.zeros((bottleneck, D))                # up-projection starts at zero
    for _ in range(epochs):
        H0 = np.tanh(Xtr @ W_frozen)                # the frozen matrix's own output
        z = H0 @ W_down
        adapter_out = z @ W_up
        H = H0 + adapter_out                         # adapter output added as a residual
        dH = 2 * (H - Htr_target) / len(Xtr)
        dWup = z.T @ dH
        dz = dH @ W_up.T
        dWdown = H0.T @ dz
        W_up -= eta * dWup; W_down -= eta * dWdown
    H0_te = np.tanh(Xte @ W_frozen)
    H_te = H0_te + (H0_te @ W_down) @ W_up
    return np.mean((H_te - Hte_target) ** 2), W_down.size + W_up.size

print(f"{'bottleneck':>11}{'trainable params':>18}{'held-out MSE':>14}")
for bn in (2, 4, 8, 16):
    loss, n_params = train_adapter(Xtr, Htr_target, Xte, Hte_target, W_pretrained,
                                   bottleneck=bn, seed=39)
    print(f"{bn:>11}{n_params:>18,}{loss:>14.6f}")

print(f"\nfor reference: LoRA at rank 4 used 2,048 params for MSE 0.130881")
print(f"unadapted {loss_frozen:.4f}, full fine-tune {loss_full:.6f} ({full_params:,} params)")
print(f"\nthe gap is structural, not a fair ranking of the two methods: this")
print(f"task was built as a linear perturbation to W, exactly what LoRA")
print(f"parameterizes directly. The adapter intervenes after the tanh")
print(f"nonlinearity, a different point in the computation, and pays for")
print(f"that mismatch here.")